<a href="https://colab.research.google.com/github/opileg/ATF/blob/Digital_Twin/digital_twin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [2]:
def calculate_probabilities(sequence):
  counts = Counter(sequence)
  total_activities = len(sequence)
  # total_count = sum(counts.values())
  probabilities = {key: value / total_activities for key, value in counts.items()}
  return probabilities

In [3]:
historical_activities = [0, 1, 2, 3, 0, 1, 3, 4, 0, 2, 1, 4, 3, 0, 1, 3, 4, 0, 1, 2, 4]
prop = calculate_probabilities(historical_activities)

activity_names = {0: 'Sleep', 1: 'Work', 2: 'Exercise', 3: 'Social', 4: 'Leisure'}
print("Activity Probabilities:")
for key, value in prop.items():
  print(f"P({activity_names[key]}) = {value:.2%}")


Activity Probabilities:
P(Sleep) = 23.81%
P(Work) = 23.81%
P(Exercise) = 14.29%
P(Social) = 19.05%
P(Leisure) = 19.05%


In [10]:
def calculate_conditional_probabilities(sequence):
  transitions = {}
  for i in range(len(sequence) - 1):
    current = sequence[i]
    next_activity = sequence[i + 1]

    if current not in transitions:
      transitions[current] = []
    transitions[current].append(next_activity)

  conditional_prob = {}
  for current, next_list in transitions.items():
    counts = Counter(next_list)
    total = len(next_list)
    conditional_prob[current] = {key: value / total for key, value in counts.items()}

  return conditional_prob

cond_probs = calculate_conditional_probabilities(historical_activities)

# Display
print("Conditional Probabilities: P(Next | Current)\n")
for current, next_probs in cond_probs.items():
    print(f"After {activity_names[current]}:")
    for next_activity, prob in sorted(next_probs.items()):
        print(f"  → {activity_names[next_activity]}: {prob:.2%}")
    print()


Conditional Probabilities: P(Next | Current)

After Sleep:
  → Work: 80.00%
  → Exercise: 20.00%

After Work:
  → Exercise: 40.00%
  → Social: 40.00%
  → Leisure: 20.00%

After Exercise:
  → Work: 33.33%
  → Social: 33.33%
  → Leisure: 33.33%

After Social:
  → Sleep: 50.00%
  → Leisure: 50.00%

After Leisure:
  → Sleep: 66.67%
  → Social: 33.33%



In [8]:
def predict_next_activity(current_activity, conditional_probs):

  """
    Predict the most likely next activity.

    Args:
        current_activity: Current activity code
        conditional_probs: Dictionary of conditional probabilities

    Returns:
        int: Most likely next activity
    """

  if current_activity not in conditional_probs:
    return None

  next_probs = conditional_probs[current_activity]
  most_likely_next = max(next_probs, key=next_probs.get)
  return most_likely_next

current = 1  # Work
predicted = predict_next_activity(current, cond_probs)
print(f"After {activity_names[current]}, most likely next: {activity_names.get(predicted, 'Unknown')}")

After Work, most likely next: Exercise


In [6]:
def sigmoid(x):
  return 1/(1 + np.exp(-x))

def relu(x):
  return np.maximum(0, x)

def tanh(x):
  return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def perceptron(inputs, weights, bias):
  weighted_sum = np.dot(inputs, weights) + bias
  output = sigmoid(weighted_sum)
  return output